# Mapping migration


## Set up

To get started on this notebook, you’ll need to restore any variables from previous notebooks to your workspace. To save time and memory, make sure to specify which variables you want to load.

In [1]:
%store -r

You will also need to import any libraries you are using in this notebook. In this notebook, you will need some additional packages to make interactive maps with vector data and access pre-defined month names.

In [ ]:
%pip install cartopy

In [ ]:
%pip install geoviews

In [2]:
### import packages
import os
import pathlib
import pandas as pd
import geopandas as gpd
import earthpy as et
from pathlib import Path
import sys
from importlib.metadata import version
from shapely.geometry import box

### libraries for dynamic mapping
import cartopy.crs as ccrs
import panel as pn
import hvplot.pandas

### get month names
import calendar

print("Imports successful!")

Imports successful!


In [3]:
### prepare a separate plotting copy in longitude/latitude
gdf = ecoregions_gdf.to_crs("EPSG:4326").copy()

### crop to the Americas and exclude polar regions
gdf = gdf.clip(box(-170, -60, -30, 80))

### simplify the geometry for display (tolerance is in degrees)
gdf["geometry"] = gdf.geometry.simplify(
    tolerance=0.05,
    preserve_topology=True,
)

### change the CRS to Web Mercator for mapping
gdf = gdf.to_crs("EPSG:3857")

### declare the actual input CRS
gdf.hvplot(
    geo=True,
    crs=ccrs.GOOGLE_MERCATOR,
)

:Polygons   [Longitude,Latitude]

### Step 5B: Map migration over time

1. If applicable, replace any variable names with the names you defined previously.
2. Replace `column_name_used_for_ecoregion_color` and `column_name_used_for_slider` with the column names you wish to use.
3. Customize your plot with your choice of title, tile source, color map, and size.

Your plot will probably still change months very slowly in your Jupyter notebook, because it calculates each month’s plot as needed. Open up the saved HTML file to see faster performance!

In [5]:
### prepare the plotting polygons
ecoreg_gdf = gdf.copy()

### make ecoregion ID the index
ecoreg_gdf = ecoreg_gdf.set_index("ECO_ID")

### combine polygons with each ecoregion/month's counts
occurrence_gdf = (
    ecoreg_gdf
    .join(occurrence_df, how="inner")
    .reset_index()
)

### check the result
occurrence_gdf.head()

,ECO_ID,month,OBJECTID,ECO_NAME,BIOME_NUM,BIOME_NAME,REALM,ECO_BIOME_,NNH,SHAPE_LENG,SHAPE_AREA,NNH_NAME,COLOR,COLOR_BIO,COLOR_NNH,LICENSE,geometry,occurrences,norm_occurrences
0,328,4,17.0,Allegheny Highlands forests,4.0,Temperate Broadleaf & Mixed Forests,Nearctic,NE04,2,24.038587,7.958751,Nature Could Reach Half Protected,#C8EBB1,#00734C,#7BC141,CC-BY 4.0,"POLYGON ((-8394924.996 5317582.703, -8405347.7...",2,0.000010
1,328,5,17.0,Allegheny Highlands forests,4.0,Temperate Broadleaf & Mixed Forests,Nearctic,NE04,2,24.038587,7.958751,Nature Could Reach Half Protected,#C8EBB1,#00734C,#7BC141,CC-BY 4.0,"POLYGON ((-8394924.996 5317582.703, -8405347.7...",3023,0.001701
2,328,6,17.0,Allegheny Highlands forests,4.0,Temperate Broadleaf & Mixed Forests,Nearctic,NE04,2,24.038587,7.958751,Nature Could Reach Half Protected,#C8EBB1,#00734C,#7BC141,CC-BY 4.0,"POLYGON ((-8394924.996 5317582.703, -8405347.7...",3242,0.002054
3,328,7,17.0,Allegheny Highlands forests,4.0,Temperate Broadleaf & Mixed Forests,Nearctic,NE04,2,24.038587,7.958751,Nature Could Reach Half Protected,#C8EBB1,#00734C,#7BC141,CC-BY 4.0,"POLYGON ((-8394924.996 5317582.703, -8405347.7...",1823,0.002214
4,328,8,17.0,Allegheny Highlands forests,4.0,Temperate Broadleaf & Mixed Forests,Nearctic,NE04,2,24.038587,7.958751,Nature Could Reach Half Protected,#C8EBB1,#00734C,#7BC141,CC-BY 4.0,"POLYGON ((-8394924.996 5317582.703, -8405347.7...",465,0.001914


In [ ]:
### activate panel and hvplot extensions you'll need for the interactive plot
pn.extension()
hv.extension("bokeh")

In [ ]:
### plot the data

### get the plot bounds so they don't change with the slider
xmin, ymin, xmax, ymax = occurrence_gdf.total_bounds

### define the slider widget
slider = pn.widgets.DiscreteSlider(
    name = 'month', 
    options = {calendar.month_name[i]: i for i in range(1, 13)}
)

### plot occurrence by ecoregion and month
migration_plot = (
    occurrence_gdf
    .hvplot(
        c = 'column_name_used_for_shape_color',
        groupby = 'column_name_used_for_slider',

        ### use background tiles
        geo = True, crs = ccrs.Mercator(), tiles = 'OSM',

        ### add a title
        title = "Your Title Here",

        ### set plot boundaries
        xlim = (xmin, xmax), ylim = (ymin, ymax),

        ### set dimensions of map
        frame_height = 600,
        frame_width = 900,

        ### put the widget at the bottom of the map
        widget_location = 'bottom',
        colorbar = False,
        widgets = {'column_name': slider}
    )
)

### wrap  plot in the Panel layout
interactive_migration_plot = pn.panel(migration_plot)

### check it out
interactive_migration_plot

Due to some annoying issues with permissions for the basemap, we can't export the map you just made as an .html file with the basemap. So we'll make another version, without a basemap, to export. We'll revisit this for your portfolio assignment to make a nice map with a basemap.

In [ ]:
### make a path for your map to live
map_path = os.path.join(data_dir, "map_name.html")

### make a version for export, without the basemap
export_migration_plot = (
    occurrence_gdf
    .hvplot(
        c = 'column_name_used_for_shape_color',
        groupby = 'column_name_used_for_slider',

        ### set CRS
        geo = True, crs = ccrs.Mercator(),

        ### add a title
        title = "Your Title Here",

        ### set plot boundaries
        xlim = (xmin, xmax), ylim = (ymin, ymax),

        ### set dimensions of map
        frame_height = 600,
        frame_width = 900,

        ### put the widget at the bottom of the map
        widget_location = 'bottom',
        colorbar = False,
        widgets = {'column_name': slider}
    )
)

### export the map as an .html file
export_migration_plot.save(
    map_path,
    embed = True,
    resources = "inline",
)

### Reflect and Respond #2: Interpret your map
What seasonal trends do you see in the GBIF observations? How does your map align with the broader literature on Veery migration?